# 03_baseline_xgb
XGBoost単体モデルのベースライン実験

In [1]:
%load_ext autoreload
%autoreload 2

import datetime
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path(
    "/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026"
)
sys.path.append(str(PROJECT_ROOT))

from common.xgboost.xgb_model import run_xgb
from common.utils.logger import get_logger
from common.utils.metrics import calculate_logloss
from common.utils.seed import seed_everything

SEED = 42
seed_everything(seed=SEED)

TARGET_COL = "10年定着ラベル"
ID_COL = "社員ID"

In [2]:
SCRIPT_NAME = "03_baseline_xgb"
TODAY = datetime.datetime.now().strftime("%Y%m%d")

LOG_DIR = PROJECT_ROOT / "logs"

logger = get_logger(SCRIPT_NAME, log_dir=str(LOG_DIR))
logger.info(f"=== [{SCRIPT_NAME}] 実験開始 ===")

OUTPUT_DIR = PROJECT_ROOT / "data" / "output" / TODAY
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSION_PATH = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}.csv"

SAVED_MODELS_DIR = PROJECT_ROOT / "saved_models" / TODAY / SCRIPT_NAME
SAVED_MODELS_DIR.mkdir(parents=True, exist_ok=True)

[2026-08-05 11:31:59] [INFO] === [03_baseline_xgb] 実験開始 ===


In [3]:
INPUT_DIR = PROJECT_ROOT / "data" / "input"

train_persona = pd.read_csv(INPUT_DIR / "employee_persona_train.csv")
test_persona = pd.read_csv(INPUT_DIR / "employee_persona_test.csv")
train_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_train.csv")
test_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_test.csv")

logger.info(f"Train Persona Shape: {train_persona.shape}, Test Persona Shape: {test_persona.shape}")
logger.info(f"Train Monthly Shape: {train_monthly.shape}, Test Monthly Shape: {test_monthly.shape}")

[2026-08-05 11:32:00] [INFO] Train Persona Shape: (2761, 20), Test Persona Shape: (2502, 19)
[2026-08-05 11:32:00] [INFO] Train Monthly Shape: (65754, 29), Test Monthly Shape: (60048, 29)


In [4]:
def transform_monthly_to_wide_all(monthly_df: pd.DataFrame) -> pd.DataFrame:
    val_cols = [col for col in monthly_df.columns if col not in ["社員ID", "経過月数"]]
    monthly_wide = monthly_df.pivot(index="社員ID", columns="経過月数", values=val_cols)
    monthly_wide.columns = [f"{col}_m{month}" for col, month in monthly_wide.columns]
    monthly_wide = monthly_wide.reset_index()
    return monthly_wide

train_monthly_wide = transform_monthly_to_wide_all(train_monthly)
test_monthly_wide = transform_monthly_to_wide_all(test_monthly)
train_df = pd.merge(train_persona, train_monthly_wide, on="社員ID", how="left")
test_df = pd.merge(test_persona, test_monthly_wide, on="社員ID", how="left")
logger.info(f"結合後 Train Shape: {train_df.shape}, Test Shape: {test_df.shape}")

[2026-08-05 11:32:00] [INFO] 結合後 Train Shape: (2761, 668), Test Shape: (2502, 667)


In [5]:
def build_features(train: pd.DataFrame, test: pd.DataFrame, target_col: str, id_col: str):
    train_proc = train.copy()
    test_proc = test.copy()
    non_num_cols = train_proc.select_dtypes(include=["object"]).columns.tolist()
    if id_col in non_num_cols:
        non_num_cols.remove(id_col)
    train_proc = train_proc.drop(columns=non_num_cols, errors="ignore")
    test_proc = test_proc.drop(columns=non_num_cols, errors="ignore")
    X_train = train_proc.drop(columns=[target_col, id_col], errors="ignore")
    y_train = train_proc[target_col]
    X_test = test_proc.drop(columns=[id_col], errors="ignore")
    input_data = {"X_train": X_train, "y_train": y_train, "X_test": X_test}
    return input_data, test_proc[id_col]

input_data, test_ids = build_features(train_df, test_df, target_col=TARGET_COL, id_col=ID_COL)
logger.info(f"X_train Shape: {input_data['X_train'].shape}")
logger.info(f"X_test Shape: {input_data['X_test'].shape}")

[2026-08-05 11:32:00] [INFO] X_train Shape: (2761, 3)
[2026-08-05 11:32:00] [INFO] X_test Shape: (2502, 3)


In [6]:
xgb_params = {
    "n_splits": 5,
    "seed": SEED,
    "save_dir": str(SAVED_MODELS_DIR),
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "early_stopping_rounds": 50,
    "verbose": False,
    "learning_rate": 0.05,
    "max_depth": 6,
    "n_estimators": 1000,
}

logger.info("XGBoostパラメータ:")
for key, value in xgb_params.items():
    logger.info(f"  {key}: {value}")

[2026-08-05 11:32:00] [INFO] XGBoostパラメータ:
[2026-08-05 11:32:00] [INFO]   n_splits: 5
[2026-08-05 11:32:00] [INFO]   seed: 42
[2026-08-05 11:32:00] [INFO]   save_dir: /Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/saved_models/20260805/03_baseline_xgb
[2026-08-05 11:32:00] [INFO]   objective: binary:logistic
[2026-08-05 11:32:00] [INFO]   eval_metric: logloss
[2026-08-05 11:32:00] [INFO]   early_stopping_rounds: 50
[2026-08-05 11:32:00] [INFO]   verbose: False
[2026-08-05 11:32:00] [INFO]   learning_rate: 0.05
[2026-08-05 11:32:00] [INFO]   max_depth: 6
[2026-08-05 11:32:00] [INFO]   n_estimators: 1000


In [7]:
logger.info("--- XGBoost トレーニング開始 ---")
xgb_res, _ = run_xgb(data=input_data, params=xgb_params)

xgb_cv = calculate_logloss(input_data["y_train"], xgb_res["oof_preds"])
logger.info(f"XGBoost CV Score (LogLoss): {xgb_cv:.4f}")

[2026-08-05 11:32:00] [INFO] --- XGBoost トレーニング開始 ---
[2026-08-05 11:32:00] [INFO] XGBoost CV Score (LogLoss): 0.6730


/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/.venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [11:32:00] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "enable_categorical", "n_estimators", "verbose" } are not used.

  warnings.warn(smsg, UserWarning)
/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/.venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [11:32:00] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "enable_categorical", "n_estimators", "verbose" } are not used.

  warnings.warn(smsg, UserWarning)
/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/.venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [11:32:00] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "enable_categorical", "n_estimators", "ve

In [8]:
sub = pd.DataFrame({ID_COL: test_ids, TARGET_COL: xgb_res["test_preds"]})
sub.to_csv(SUBMISSION_PATH, index=False)
logger.info(f"提出ファイルを保存しました: {SUBMISSION_PATH}")
logger.info("=== 実験完了 ===")

print(f"\n提出ファイル: {SUBMISSION_PATH}")
print(f"CV Score (LogLoss): {xgb_cv:.4f}")

[2026-08-05 11:32:00] [INFO] 提出ファイルを保存しました: /Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/data/output/20260805/20260805_03_baseline_xgb.csv
[2026-08-05 11:32:00] [INFO] === 実験完了 ===

提出ファイル: /Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/data/output/20260805/20260805_03_baseline_xgb.csv
CV Score (LogLoss): 0.6730
